# BEATs (Job)
Ik heb de unilm/beats repo gebruikt voor de beats, dit is alsnog diegene van de challenge. Voor de imports gebruik ik deze. Je moet deze zelf clonen https://github.com/microsoft/unilm/tree/master/beats. Ook heel belangrijk is dat je de BEATs_iter3_plus_AS2M.pt file hebt gedownload (kan via de readme van de bovenstaande github repo)

In [18]:
import os
import sys
import torch
import torchaudio
import numpy as np
from scipy.io import wavfile
import torch.nn as nn

sys.path.append(r"C:\Users\20223669\OneDrive - TU Eindhoven\Documents\GitHub\Team-Internship-Sorama\unilm\beats")
from BEATs import BEATs, BEATsConfig

## Load model
Hier wordt het model geladen (vanuit de git repository die net gecloned is) en vervolgens wordt de 'checkpoint' geladen. Dit zijn de getrainde weights. Ook is het handig als je CUDA hebt, dan gebruikt je laptop je GPU ipv je CPU. Dit is net wat sneller maar wel wat werk om aan de praat te krijgen. 

In [ ]:
checkpoint_path = r"C:\Users\20223669\OneDrive - TU Eindhoven\Documents\GitHub\Team-Internship-Sorama\BEATs_iter3_plus_AS2M.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu")
cfg = BEATsConfig(checkpoint["cfg"])  # Wrap dict in config object

# Initialize BEATs
model = BEATs(cfg)
model.load_state_dict(checkpoint["model"])
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

c:\Users\20223669\AppData\Local\miniconda3\envs\tfgpu\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


False


## Prepocess data
Hier zou data augmentatie aan toegevoegd kunnen worden

In [15]:
def augment_waveform(waveform, sample_rate, noise_prob=0.5, gain_prob=0.5, shift_prob=0.5):
    # Additive Gaussian noise
    if torch.rand(1).item() < noise_prob:
        noise_std = torch.empty(1).uniform_(0.001, 0.01).item()
        waveform = waveform + noise_std * torch.randn_like(waveform)

    # Random gain
    if torch.rand(1).item() < gain_prob:
        gain = torch.empty(1).uniform_(0.8, 1.2).item()
        waveform = waveform * gain

    # Random time shift (up to 100 ms)
    if torch.rand(1).item() < shift_prob:
        max_shift = int(0.1 * sample_rate)
        shift = torch.randint(-max_shift, max_shift + 1, (1,)).item()
        waveform = torch.roll(waveform, shifts=shift, dims=0)

    # Keep values in safe range
    waveform = torch.clamp(waveform, -1.0, 1.0)
    return waveform


def _load_wav_with_scipy(file_path):
    sr, data = wavfile.read(file_path)
    data = np.asarray(data)

    if np.issubdtype(data.dtype, np.integer):
        max_val = np.iinfo(data.dtype).max
        data = data.astype(np.float32) / float(max_val)
    else:
        data = data.astype(np.float32)

    if data.ndim == 1:
        waveform = torch.from_numpy(data).unsqueeze(0)  # [1, time]
    else:
        waveform = torch.from_numpy(data.T)  # [channels, time]

    return waveform, int(sr)


def load_audio(file_path, target_sr=16000, apply_augment=False):
    try:
        waveform, sr = torchaudio.load(file_path)
    except RuntimeError:
        waveform, sr = _load_wav_with_scipy(file_path)

    # Mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0)
    else:
        waveform = waveform.squeeze(0)  # Make sure it's 1D

    # Resample
    if sr != target_sr:
        waveform = torchaudio.functional.resample(waveform.unsqueeze(0), sr, target_sr).squeeze(0)

    # Pad short waveforms to 400 samples (25ms)
    if waveform.shape[0] < 400:
        pad_len = 400 - waveform.shape[0]
        waveform = torch.nn.functional.pad(waveform, (0, pad_len))

    if apply_augment:
        waveform = augment_waveform(waveform, target_sr)

    return waveform


def extract_beats_embedding(waveform):
    waveform = waveform.to(device)
    with torch.no_grad():
        features, _ = model.extract_features(waveform.unsqueeze(0))  # [1, T, 768]
        clip_embedding = features.mean(dim=1).squeeze(0).cpu().numpy()  # [768]
    return clip_embedding

## Embeddings uit 1 machine halen (duurt ong 10 min)
Nu heeft de BEATs model 768 features voor iedere clip gehaald. Deze features kun je in een ander model stoppen om hem uiteindelijk te classifyen. 

In [17]:
input_folder = r"C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\bearing\train"
output_folder = r"C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\features"
os.makedirs(output_folder, exist_ok=True)

# Turn this off for validation/test feature extraction
APPLY_AUGMENTATION = True

for root, _, files in os.walk(input_folder):
    for file in files:
        if file.endswith(".wav"):
            file_path = os.path.join(root, file)
            waveform = load_audio(file_path, apply_augment=APPLY_AUGMENTATION)
            embedding = extract_beats_embedding(waveform)

            # Mirror folder structure in output
            relative_path = os.path.relpath(root, input_folder)
            out_dir = os.path.join(output_folder, relative_path)
            os.makedirs(out_dir, exist_ok=True)

            out_path = os.path.join(out_dir, os.path.splitext(file)[0] + ".npy")
            np.save(out_path, embedding)

            print(f"Processed: {file_path} -> {out_path}")

print("All embeddings extracted successfully!")

Processed: C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\bearing\train\section_00_source_train_normal_0000_noAttribute.wav -> C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\features\.\section_00_source_train_normal_0000_noAttribute.npy
Processed: C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\bearing\train\section_00_source_train_normal_0001_noAttribute.wav -> C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\features\.\section_00_source_train_normal_0001_noAttribute.npy
Processed: C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\bearing\train\section_00_source_train_normal_0002_noAttribute.wav -> C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\features\.\section_00_source_train_normal_0002_noAttribute.npy
Processed: C:\Users\20223669\OneDrive

KeyboardInterrupt: 

## Example classifyer head

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ====== 1️⃣ Paths ======
features_folder = r"C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\features"
output_csv = r"C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\anomaly_scores.csv"

# ====== 2️⃣ Load all embeddings ======
all_files = []
for root, _, files in os.walk(features_folder):
    for file in files:
        if file.endswith(".npy"):
            all_files.append(os.path.join(root, file))

embeddings_list = []
file_paths = []
for f in all_files:
    emb = np.load(f)
    embeddings_list.append(emb)
    file_paths.append(f)

X = np.stack(embeddings_list)  # [N, 768]
X = torch.tensor(X, dtype=torch.float32)
print("Loaded embeddings shape:", X.shape)

# ====== 3️⃣ Define Autoencoder ======
class AE(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

# ====== 4️⃣ Prepare DataLoader ======
batch_size = 32
dataset = TensorDataset(X)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# ====== 5️⃣ Initialize model ======
device = "cuda" if torch.cuda.is_available() else "cpu"
ae = AE().to(device)
optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# ====== 6️⃣ Train Autoencoder ======
epochs = 50
for epoch in range(epochs):
    epoch_loss = 0
    for batch in loader:
        x_batch = batch[0].to(device)
        optimizer.zero_grad()
        x_hat = ae(x_batch)
        loss = loss_fn(x_hat, x_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * x_batch.size(0)
    epoch_loss /= len(loader.dataset)
    if epoch % 10 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch+1}/{epochs} Loss: {epoch_loss:.6f}")

# ====== 7️⃣ Compute anomaly scores ======
ae.eval()
scores = []
with torch.no_grad():
    for emb, path in zip(X, file_paths):
        emb = emb.to(device)
        recon = ae(emb)
        score = torch.norm(emb - recon).item()  # L2 reconstruction error
        scores.append((path, score))

# ====== 8️⃣ Save anomaly scores ======
import csv
with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["file", "anomaly_score"])
    writer.writerows(scores)

print(f"✅ Anomaly scores saved to {output_csv}")